In [17]:
from dotenv import load_dotenv
from huggingface_hub import login
import torch
from datasets import load_dataset
from tqdm import tqdm
import os

# Silence warnings and clean up output
import transformers
from transformers import pipeline
transformers.logging.set_verbosity_error() 
transformers.utils.logging.disable_progress_bar()
import warnings, logging

from huggingface_hub.utils import disable_progress_bars
disable_progress_bars() 

load_dotenv()

login(token=os.getenv("HF_TOKEN"))

device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


cuda


In [18]:
# Test run 
pipe = pipeline(
    "image-text-to-text",
    model="llava-hf/llava-1.5-7b-hf",
    device=device,
    dtype=torch.float16,
)

pipe.model.generation_config.max_length = None

messages = [{
    "role": "user",
    "content": [
        {"type": "image", "url": "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/transformers/tasks/ai2d-demo.jpg"},
        {"type": "text", "text": "What does the label 15 represent? (1) lava (2) core (3) tunnel (4) ash cloud"},
    ],
}]

out = pipe(text=messages, max_new_tokens=20)
print(out[0]["generated_text"])

[{'role': 'user', 'content': [{'type': 'image', 'url': 'https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/transformers/tasks/ai2d-demo.jpg'}, {'type': 'text', 'text': 'What does the label 15 represent? (1) lava (2) core (3) tunnel (4) ash cloud'}]}, {'role': 'assistant', 'content': ' The label 15 represents the ash cloud, which is a natural phenomenon associated with volcan'}]


In [19]:
# CUB-200 Birds dataset

dataset = load_dataset("cassiekang/cub200_dataset")
print(dataset)
print(dataset["test"].features) 

def make_messages(img):
    return [{
        "role": "user",
        "content": [
            {"type": "image", "image": img},
            {"type": "text", "text": "Identify the bird in the photo"},
        ],
    }]

def run(row):
    return pipe(text=make_messages(row["image"]), max_new_tokens=60,
                return_full_text=False)[0]["generated_text"]

run(dataset["test"][0])

for i in tqdm(range(3)):
    row = dataset["train"][i]
    pred = run(row)
    tqdm.write(f"[{i}] Pred:  {pred.strip()}\n[{i}] Truth: {row['text']}\n")

DatasetDict({
    train: Dataset({
        features: ['text', 'image'],
        num_rows: 5994
    })
    test: Dataset({
        features: ['text', 'image'],
        num_rows: 5794
    })
})
{'text': Value('string'), 'image': Image(mode=None, decode=True)}


 33%|███▎      | 1/3 [00:00<00:00,  7.15it/s]

[0] Pred:  The bird in the photo is a seagull.
[0] Truth: A photo of a laysan albatross



100%|██████████| 3/3 [00:00<00:00,  5.50it/s]

[1] Pred:  The bird in the photo is a small, gray and white bird, possibly a sparrow.
[1] Truth: A photo of a sage thrasher

[2] Pred:  The bird in the photo is a small brown and white bird, possibly a sparrow.
[2] Truth: A photo of a bank swallow

